# 👥 View — Clientes por Região

Validação da view `vw_clientes_regiao` antes de mover para o Streamlit.

In [2]:
import pandas as pd
import sys
sys.path.append('..')

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos  = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
clientes = pd.read_csv("../dados/clientes_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [3]:
    # ==========================================
    # 1. MERGE DAS TABELAS
    # ==========================================
    # Une pedidos e clientes pelo customer_id.
    # O LEFT JOIN preserva todos os pedidos, mesmo
    # que algum registro de cliente não seja encontrado.
df = pedidos.merge(
        clientes,
        on='customer_id',
        how='left'
    )


In [12]:
pedidos['order_id'].nunique()

99441

In [13]:
pedidos.groupby('order_status').size().sort_values(ascending=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
dtype: int64

In [9]:
    # ==========================================
    # 2. TRADUÇÃO DOS STATUS
    # ==========================================
status_traducao = {
        'delivered':   'Entregue',
        'shipped':     'Em Transporte',
        'canceled':    'Cancelado',
        'unavailable': 'Indisponível',
        'invoiced':    'Faturado',
        'processing':  'Em Processamento',
        'created':     'Criado',
        'approved':    'Aprovado'
    }
df['status_pt'] = (
        df['order_status']
        .map(status_traducao)
        .fillna('Outros')
    )

    # ==========================================
    # 3. DIMENSÕES DE TEMPO
    # ==========================================

#Cria as dimensões utilizadas nas análises
# temporais da página.
df['ano'] = (df['order_purchase_timestamp'].dt.year)

df['mes'] = (df['order_purchase_timestamp'].dt.month)

df['data_mes'] = (df['order_purchase_timestamp'].dt.to_period('M').astype(str))

# ==========================================
    # 4. AGREGAÇÃO ANALÍTICA
    # ==========================================
    # Granularidade:
    # Estado × Cidade × Ano × Mês × Status
    #
    # total_clientes:
    # quantidade de clientes únicos (customer_unique_id)
    # associados a pedidos naquela granularidade.
    #
    # total_pedidos:
    # quantidade de pedidos únicos naquela
    # mesma granularidade.

clientes_regiao = (
        df.groupby(
            [
                'customer_state',
                'customer_city',
                'ano',
                'mes',
                'data_mes',
                'status_pt'
            ]
        ).agg(total_clientes=('customer_unique_id','nunique'),
              total_pedidos=('order_id','nunique')).reset_index()
    )



In [10]:
# ==============
# 5. ORDENAÇÃO FINAL
# ==========================================
# Ordena os registros pela quantidade de clientes.

clientes_regiao = clientes_regiao.sort_values(
'total_clientes',
ascending=False
)

clientes_regiao.head()

,customer_state,customer_city,ano,mes,data_mes,status_pt,total_clientes,total_pedidos
22554,SP,sao paulo,2018,8,2018-08,Entregue,1261,1269
22539,SP,sao paulo,2018,5,2018-05,Entregue,1176,1191
22533,SP,sao paulo,2018,4,2018-04,Entregue,1129,1135
22527,SP,sao paulo,2018,3,2018-03,Entregue,1111,1127
22505,SP,sao paulo,2017,11,2017-11,Entregue,1060,1081


### 🔎 Validando a função da view

In [11]:


from views.vw_clientes_regiao import get_vw_clientes_regiao

df_clientes = get_vw_clientes_regiao(
pedidos,
clientes
)

df_clientes.head()

,customer_state,customer_city,ano,mes,data_mes,status_pt,total_clientes,total_pedidos
22554,SP,sao paulo,2018,8,2018-08,Entregue,1261,1269
22539,SP,sao paulo,2018,5,2018-05,Entregue,1176,1191
22533,SP,sao paulo,2018,4,2018-04,Entregue,1129,1135
22527,SP,sao paulo,2018,3,2018-03,Entregue,1111,1127
22505,SP,sao paulo,2017,11,2017-11,Entregue,1060,1081
